# Préparation des datasets d'entraînement

**Objectif** : construire les datasets qui serviront à la modélisation, avec les bonnes lignes et les bonnes variables, sans fuite de données.

| Service | Source | Cible |
|---|---|---|
| `/predict` | `data/agriculture-crop-yield/crop_yield.csv` | `Yield_tons_per_hectare` |
| `/recommend` | `data/processed/crop_yield_clean.csv` (notebook 04) | `yield_t_ha` |

Pas de feature engineering, d'encodage, de standardisation ni d'imputation ici : ces étapes relèvent de la
modélisation.

## Imports

In [1]:
import pandas as pd

from agritech.config import AGRICULTURE_CROP_YIELD_FILENAME, PATHS

# 1. `/predict` — Agriculture CropYield

## Lecture et contrôles de base

In [2]:
csv_path = PATHS.data_agriculture_crop_yield / AGRICULTURE_CROP_YIELD_FILENAME
df_agri = pd.read_csv(csv_path)

print("lignes x colonnes  :", df_agri.shape)
print("valeurs manquantes :", df_agri.isna().sum().sum())
print("doublons stricts   :", df_agri.duplicated().sum())
print("\ntypes :")
print(df_agri.dtypes.to_string())

lignes x colonnes  : (1000000, 10)
valeurs manquantes : 0
doublons stricts   : 0

types :
Region                     object
Soil_Type                  object
Crop                       object
Rainfall_mm               float64
Temperature_Celsius       float64
Fertilizer_Used              bool
Irrigation_Used              bool
Weather_Condition          object
Days_to_Harvest             int64
Yield_tons_per_hectare    float64


**Observations :**

- Aucune valeur manquante, aucun doublon, types corrects.

## Cible : rendements négatifs

Un rendement négatif est impossible. Les autres valeurs atypiques sont gardées.

In [3]:
negatifs = df_agri[df_agri["Yield_tons_per_hectare"] < 0].copy()
print(f"rendements négatifs exclus : {len(negatifs)} ({len(negatifs) / len(df_agri):.3%})")

rendements négatifs exclus : 231 (0.023%)


**Observations :**

- Les 231 rendements négatifs sont exclus de l’entraînement et sauvegardés à part pour un test exploratoire après modélisation.
- Leurs cibles invalides ne serviront pas à évaluer les performances.

## Variables conservées pour la modélisation

Les neuf variables d’entrée sont conservées. Leur utilité et leur disponibilité au moment de prédire seront examinées lors de la modélisation.

In [4]:
FEATURES_PREDICT = [
    "Crop",
    "Soil_Type",
    "Rainfall_mm",
    "Temperature_Celsius",
    "Fertilizer_Used",
    "Irrigation_Used",
    "Region",
    "Weather_Condition",
    "Days_to_Harvest",
]
CIBLE_PREDICT = "Yield_tons_per_hectare"

df_predict = df_agri[df_agri[CIBLE_PREDICT] >= 0]

X_predict = df_predict[FEATURES_PREDICT]
y_predict = df_predict[CIBLE_PREDICT]

print(f"lignes           : {len(X_predict)} (sur {len(df_agri)}, soit {len(df_agri) - len(X_predict)} retirées)")
print(f"features         : {FEATURES_PREDICT}")
print(f"cible            : {CIBLE_PREDICT}")
print(f"manquants dans X : {X_predict.isna().sum().sum()}")
X_predict.head()

lignes           : 999769 (sur 1000000, soit 231 retirées)
features         : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Region', 'Weather_Condition', 'Days_to_Harvest']
cible            : Yield_tons_per_hectare
manquants dans X : 0


,Crop,Soil_Type,Rainfall_mm,Temperature_Celsius,Fertilizer_Used,Irrigation_Used,Region,Weather_Condition,Days_to_Harvest
0,Cotton,Sandy,897.077239,27.676966,False,True,West,Cloudy,122
1,Rice,Clay,992.673282,18.026142,True,True,South,Rainy,140
2,Barley,Loam,147.998025,29.794042,False,False,North,Sunny,106
3,Soybean,Sandy,986.866331,16.644190,False,True,North,Rainy,146
4,Wheat,Silt,730.379174,31.620687,True,True,South,Cloudy,110


**Observations :**

- 999 769 lignes, 9 variables d’entrée, aucune valeur manquante.
- Les variables catégorielles restent en texte : l’encodage sera appris sur le train.

# 2. `/recommend` — dataset historique nettoyé

2013 est réservée au test final. Les variables sont gardées telles quelles : le feature engineering sera
traité dans un notebook dédié.

## Lecture et contrôles de base

In [5]:
CLE = ["iso3", "year", "crop"]
df_hist = pd.read_csv(PATHS.data_processed / "crop_yield_clean.csv")

print("lignes x colonnes   :", df_hist.shape)
print("doublons sur la clé :", df_hist.duplicated(CLE).sum())
print("cultures / pays     :", df_hist.crop.nunique(), "/", df_hist.iso3.nunique())
print("années              :", df_hist.year.min(), "-", df_hist.year.max())
print("valeurs manquantes  :", df_hist.isna().sum().sum())
print("rendements <= 0     :", (df_hist.yield_t_ha <= 0).sum())

assert len(df_hist) == 16_319
assert not df_hist.duplicated(CLE).any()
assert df_hist.isna().sum().sum() == 0

lignes x colonnes   : (16319, 8)
doublons sur la clé : 0
cultures / pays     : 10 / 115
années              : 1990 - 2013
valeurs manquantes  : 0
rendements <= 0     : 0


**Observations :**

- `crop_yield_clean.csv` : 16 319 lignes, 115 pays, 10 cultures, 1990-2013.
- Aucune valeur manquante, aucun rendement nul ou négatif.
- Le nettoyage a été fait dans le notebook 04 : il n'est pas refait ici.

## Les 10 cultures

Nombre de lignes, de pays et d'années par culture.

In [6]:
couverture_cultures = df_hist.groupby("crop").agg(
    lignes=("yield_t_ha", "size"),
    pays=("iso3", "nunique"),
    années=("year", "nunique"),
    rendement_médian=("yield_t_ha", "median"),
).sort_values("lignes", ascending=False)

couverture_cultures.round(2)

,lignes,pays,années,rendement_médian
crop,,,,
Maize,2505,107,24,2.55
Potatoes,2498,107,24,15.88
Wheat,2180,93,24,2.45
"Rice, paddy",1895,81,24,3.47
Sorghum,1788,79,24,1.26
Soybeans,1657,74,24,1.58
Sweet potatoes,1424,60,24,8.18
Cassava,1176,49,24,10.00
Plantains and others,628,27,24,8.34


**Observations :**

- Les 10 cultures sont présentes sur les 24 années.
- L'igname (568 lignes, 24 pays) et le plantain (628 lignes, 27 pays) ont le moins de données.
- Rendement médian de 1,3 t/ha (sorgho) à 16 t/ha (pomme de terre) : les tubercules arriveront en
  tête d'un classement en t/ha.

## Conditions observées avant 2013

Quantiles 5 % et 95 % calculés hors test final, sur les années avant 2013. Ces repères décrivent les données, pas la fiabilité du modèle. Pour une règle évaluée en validation, ils devront être recalculés sur le train de chaque découpage.

In [7]:
donnees_avant_test = df_hist[df_hist["year"] < 2013]
domaine = donnees_avant_test.groupby("crop").agg(
    temp_p5=("avg_temp", lambda s: s.quantile(0.05)),
    temp_p95=("avg_temp", lambda s: s.quantile(0.95)),
    pluie_p5=("rain_mm", lambda s: s.quantile(0.05)),
    pluie_p95=("rain_mm", lambda s: s.quantile(0.95)),
)

domaine.sort_values("temp_p5").round(0)

,temp_p5,temp_p95,pluie_p5,pluie_p95
crop,,,,
Wheat,6.0,27.0,89.0,1996.0
Potatoes,6.0,28.0,92.0,2274.0
Maize,8.0,28.0,92.0,2387.0
Soybeans,8.0,27.0,250.0,2280.0
Sorghum,9.0,28.0,92.0,2280.0
"Rice, paddy",9.0,28.0,151.0,2702.0
Sweet potatoes,11.0,28.0,171.0,2697.0
Yams,16.0,28.0,282.0,3142.0
Plantains and others,17.0,28.0,1071.0,2387.0


**Observations :**

- Ces intervalles couvrent les 90 % centraux de chaque variable, par culture, avant 2013.
- Une valeur extérieure est inhabituelle dans ces données, sans prouver que la prédiction est peu fiable.

## Variables candidates

| Variable | Rôle | Pourquoi |
|---|---|---|
| `crop` | variable candidate | culture à classer |
| `avg_temp` | variable candidate | température moyenne annuelle du pays |
| `rain_mm` | variable candidate | valeur de pluie fixe par pays dans le fichier |
| `pesticides_t` | variable candidate | tonnage national de pesticides de l'année |
| `iso3` | variable candidate à tester | code du pays : identifie le pays et son contexte ; son apport est mesuré dans le notebook de modélisation `/recommend` |
| `year` | variable candidate à tester | séparer les années : avant 2013 pour l'entraînement et la validation, 2013 pour le test final ; son apport est mesuré de la même façon |
| `area` | contexte hors modèle | nom du pays, pour les analyses |

Les variables sont gardées telles quelles : aucune feature n'est construite ici.

In [8]:
# variables candidates : le notebook de modélisation /recommend mesure l'apport de iso3 et year
FEATURES_RECOMMEND = ["iso3", "year", "crop", "avg_temp", "rain_mm", "pesticides_t"]
CIBLE_RECOMMEND = "yield_t_ha"
CONTEXTE_RECOMMEND = ["area"]   # hors modèle

# le dataset historique nettoyé est gardé en entier : aucune ligne retirée
df_recommend = df_hist
X_recommend = df_recommend[FEATURES_RECOMMEND]
y_recommend = df_recommend[CIBLE_RECOMMEND]
contexte_recommend = df_recommend[CONTEXTE_RECOMMEND]

print(f"lignes               : {len(X_recommend)}")
print(f"variables candidates : {FEATURES_RECOMMEND}")
print(f"contexte hors modèle : {CONTEXTE_RECOMMEND}")
print(f"cible                : {CIBLE_RECOMMEND}")
print(f"manquants dans X     : {X_recommend.isna().sum().sum()}")
X_recommend.head()

lignes               : 16319
variables candidates : ['iso3', 'year', 'crop', 'avg_temp', 'rain_mm', 'pesticides_t']
contexte hors modèle : ['area']
cible                : yield_t_ha
manquants dans X     : 0


,iso3,year,crop,avg_temp,rain_mm,pesticides_t
0,ALB,1990,Maize,16.37,1485.0,121.0
1,ALB,1991,Maize,15.36,1485.0,121.0
2,ALB,1992,Maize,16.06,1485.0,121.0
3,ALB,1993,Maize,16.05,1485.0,121.0
4,ALB,1994,Maize,16.96,1485.0,201.0


**Observations :**

- 16 319 lignes, aucune valeur manquante : le dataset historique nettoyé est gardé en entier.
- 6 variables candidates : `iso3`, `year`, `crop`, `avg_temp`, `rain_mm` et `pesticides_t`. Le notebook de
  modélisation `/recommend` mesure l'apport de `iso3` et `year`.
- `area` reste dans le fichier comme contexte hors modèle.

# 3. Sauvegarde

Deux datasets d’entraînement et un fichier de rendements négatifs dans `data/processed/`, non versionnés et reconstruits par ce notebook. Pas d'encodage
ni de standardisation : ces étapes seront apprises sur le train pendant la modélisation. Le fichier
`/recommend` garde `iso3`, `area` et `year` pour identifier le pays, séparer les années
et faire des analyses ; `iso3` et `year` seront aussi testés comme variables candidates.
Le fichier des rendements négatifs est réservé à l’examen des prédictions, hors entraînement et évaluation.

In [9]:
PATHS.data_processed.mkdir(parents=True, exist_ok=True)
chemin_predict = PATHS.data_processed / "predict_training_dataset.csv"
chemin_recommend = PATHS.data_processed / "recommend_training_dataset.csv"

df_predict[FEATURES_PREDICT + [CIBLE_PREDICT]].to_csv(chemin_predict, index=False)
# ordre des colonnes du fichier /recommend : pays et année d'abord, puis les autres variables et la cible
COLONNES_RECOMMEND = ["iso3", "area", "year", "crop", "avg_temp", "rain_mm", "pesticides_t", CIBLE_RECOMMEND]
assert sorted(COLONNES_RECOMMEND) == sorted(CONTEXTE_RECOMMEND + FEATURES_RECOMMEND + [CIBLE_RECOMMEND])
df_recommend[COLONNES_RECOMMEND].to_csv(chemin_recommend, index=False)

chemin_negatifs = PATHS.data_processed / "predict_negative_yield_rows.csv"
negatifs.to_csv(chemin_negatifs, index=False)
print("écrit :", chemin_negatifs.relative_to(PATHS.root))

# chemins relatifs : pas de chemin local enregistré dans le notebook
print("écrit :", chemin_predict.relative_to(PATHS.root))
print("écrit :", chemin_recommend.relative_to(PATHS.root))

écrit : data/processed/predict_negative_yield_rows.csv
écrit : data/processed/predict_training_dataset.csv
écrit : data/processed/recommend_training_dataset.csv


In [10]:
# Relecture des deux fichiers et contrôles
relu_predict = pd.read_csv(chemin_predict)
relu_recommend = pd.read_csv(chemin_recommend)

assert len(relu_predict) == 999_769
assert relu_predict.columns.tolist() == FEATURES_PREDICT + [CIBLE_PREDICT]
assert relu_predict.isna().sum().sum() == 0
assert (relu_predict[CIBLE_PREDICT] >= 0).all()

# /recommend : le fichier relu est le dataset historique nettoyé, colonnes choisies, sans ligne perdue
pd.testing.assert_frame_equal(relu_recommend, df_recommend[COLONNES_RECOMMEND].reset_index(drop=True))
assert len(relu_recommend) == len(df_hist)
assert relu_recommend.columns.tolist() == COLONNES_RECOMMEND
assert relu_recommend["iso3"].nunique() == df_hist["iso3"].nunique()
assert relu_recommend["crop"].nunique() == df_hist["crop"].nunique()
assert (relu_recommend["year"].min(), relu_recommend["year"].max()) == (df_hist["year"].min(), df_hist["year"].max())
assert not relu_recommend.duplicated(["iso3", "year", "crop"]).any()
assert relu_recommend.isna().sum().sum() == 0
assert (relu_recommend[CIBLE_RECOMMEND] > 0).all()

print("predict   :", relu_predict.shape, "| colonnes :", relu_predict.columns.tolist())
print("recommend :", relu_recommend.shape, "| colonnes :", relu_recommend.columns.tolist())
print(f"            {relu_recommend['iso3'].nunique()} pays, {relu_recommend['crop'].nunique()} cultures, "
      f"{relu_recommend['year'].min()}-{relu_recommend['year'].max()}, clé unique, 0 NaN")
print("contrôles : OK")

relu_negatifs = pd.read_csv(chemin_negatifs)
assert len(relu_negatifs) == len(negatifs) == 231
assert relu_negatifs.columns.tolist() == df_agri.columns.tolist()
assert (relu_negatifs[CIBLE_PREDICT] < 0).all()

predict   : (999769, 10) | colonnes : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Region', 'Weather_Condition', 'Days_to_Harvest', 'Yield_tons_per_hectare']
recommend : (16319, 8) | colonnes : ['iso3', 'area', 'year', 'crop', 'avg_temp', 'rain_mm', 'pesticides_t', 'yield_t_ha']
            115 pays, 10 cultures, 1990-2013, clé unique, 0 NaN
contrôles : OK


# Conclusion
- `/predict` : 999 769 lignes, neuf variables d’entrée conservées ; 231 rendements négatifs sauvegardés séparément.
- `/recommend` : 16 319 lignes, soit tout le dataset historique nettoyé, avec les variables d'origine `avg_temp`, `rain_mm` et `pesticides_t` ; le feature engineering est traité à part.
- `iso3` et `year` sont conservés et seront testés comme variables candidates dans le notebook de modélisation `/recommend` ; `area` reste un contexte hors modèle. 2013 est réservée au test final.

**Limites :** données historiques nationales, conditions inhabituelles à signaler, rendement prédit différent de la rentabilité.